# LL Gesamtnotebook

Dieses Notebook enthält den gesamten benutzbaren Code von Linus Lauschke. Wiederholungen, Fehler etc wurden entfernt um den Umfang zu reduzieren.

## Import

In [ ]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from optuna_integration import XGBoostPruningCallback, CatBoostPruningCallback

import joblib

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframe

In [ ]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

pd.Series({
    "Zeilen": len(df_raw),
    "Spalten": df_raw.shape[1],
    "Positivrate target": df_raw["target"].astype(int).mean(),
})

### Splits erstellen

In [ ]:
n = len(df_raw)
idx_all = np.arange(n)

n, len(idx_all)

In [ ]:
idx_trainval, idx_test = train_test_split(
    idx_all,
    test_size=0.1,
    stratify=df_raw["target"],
    random_state=42
)
len(idx_trainval) + len(idx_test)

In [ ]:
idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=(0.1/0.9), # 10% von den 100% raw und nicht von den 90% trainval
    stratify=df_raw["target"].iloc[idx_trainval],
    random_state=42
)

In [ ]:
for idx, name in zip([idx_test, idx_val, idx_train],["test", "val", "test"]):
    print(f'{name} : {len(idx)} --> {len(idx)/n:.2%}')

In [ ]:
# speicherung auskommentiert, da schon besteht
#np.save("../../../data/processed/train_idx.npy", idx_train)
#np.save("../../../data/processed/test_idx.npy", idx_test)
#np.save("../../../data/processed/val_idx.npy", idx_val)

### Splits laden

In [ ]:
idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

## Vorbereitung & Hilfsfunktion

In [ ]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]
x_full, y_full = df_raw[feat_cols], df_raw["target"]

idx_train_val = np.concatenate([idx_train, idx_val])
df_train_val = df_raw.iloc[idx_train_val]
x_train_val, y_train_val = df_train_val[feat_cols], df_train_val["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)],
        "full": [len(df_raw), len(x_full), len(y_full)],
        "train_val": [len(df_train_val), len(x_train_val), len(y_train_val)]
    }
)

In [ ]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]
feat_cols_no_calc = [c for c in feat_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

In [ ]:
def eval_modell(mod_idx, model, x_train, y_train, x_val, y_val, train_time, best_iter=None):
    """AUC auf Train und Val"""
    auc_train = roc_auc_score(y_train, model.predict_proba(x_train)[:, 1])
    auc_val = roc_auc_score(y_val, model.predict_proba(x_val)[:, 1])
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_test": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "best_iter": best_iter,
        "trainingszeit": train_time
    }

In [ ]:
def mv_for_cb(df, fill_var="missing"):
    """df copy mit ersetzten nans"""
    outs = df.copy()
    for col in outs.columns:
        if outs[col].dtype.name == "category":
            if fill_var not in outs[col].cat.categories:
                outs[col] = outs[col].cat.add_categories([fill_var])
            outs[col] = outs[col].fillna(fill_var)
    return outs

In [ ]:
x_train_cb = mv_for_cb(x_train)
x_val_cb = mv_for_cb(x_val)
x_test_cb = mv_for_cb(x_test)
x_full_cb = mv_for_cb(x_full)
x_train_val_cb = mv_for_cb(x_train_val)

x_train_no_calc_cb = x_train_cb[feat_cols_no_calc]
x_val_no_calc_cb = x_val_cb[feat_cols_no_calc]
x_test_no_calc_cb = x_test_cb[feat_cols_no_calc]
x_full_no_calc_cb = x_full_cb[feat_cols_no_calc]
x_train_val_no_calc_cb = x_train_val_cb[feat_cols_no_calc]

pd.Series(
    {
        "train": [len(x_train_cb), len(x_train_no_calc_cb)],
        "val": [len(x_val_cb), len(x_val_no_calc_cb)],
        "test": [len(x_test_cb), len(x_test_no_calc_cb)],
        "full": [len(x_full_cb), len(x_full_no_calc_cb)],
        "train_val": [len(x_train_val_cb), len(x_train_val_no_calc_cb)]
    }
)

In [ ]:
x_train_no_calc = x_train[feat_cols_no_calc]
x_val_no_calc = x_val[feat_cols_no_calc]
x_test_no_calc = x_test[feat_cols_no_calc]

In [ ]:
targ_enc = TargetEncoder(cv=5, random_state=RANDOM_STATE)

x_train_te = x_train.copy()
x_val_te = x_val.copy()
x_test_te = x_test.copy()

x_train_te[high_kard_cols] = targ_enc.fit_transform(x_train[high_kard_cols], y_train)
x_val_te[high_kard_cols] = targ_enc.transform(x_val[high_kard_cols])
x_test_te[high_kard_cols] = targ_enc.transform(x_test[high_kard_cols])

x_train_no_calc_te = x_train_te[feat_cols_no_calc]
x_val_no_calc_te = x_val_te[feat_cols_no_calc]
x_test_no_calc_te = x_test_te[feat_cols_no_calc]

x_train_te[high_kard_cols].describe()

In [ ]:
targ_enc_hgb = TargetEncoder(
    cv=5,
    shuffle=True,
    random_state=RANDOM_STATE,
    smooth=50.0
)

te_train_hgb = pd.DataFrame(
    targ_enc_hgb.fit_transform(x_train[cat_cols], y_train),
    columns=cat_cols,
    index=x_train.index
)

te_val_hgb = pd.DataFrame(
    targ_enc_hgb.transform(x_val[cat_cols]),
    columns=cat_cols,
    index=x_val.index
)

te_test_hgb = pd.DataFrame(
    targ_enc_hgb.transform(x_test[cat_cols]),
    columns=cat_cols,
    index=x_test.index
)

In [ ]:
x_train_hgb = x_train.copy()
x_val_hgb = x_val.copy()
x_test_hgb = x_test.copy()

x_train_hgb[cat_cols] = te_train_hgb
x_val_hgb[cat_cols] = te_val_hgb
x_test_hgb[cat_cols] = te_test_hgb

x_train_hgb_no_calc = x_train_hgb[feat_cols_no_calc]
x_val_hgb_no_calc = x_val_hgb[feat_cols_no_calc]
x_test_hgb_no_calc = x_test_hgb[feat_cols_no_calc]

## EDA

In [ ]:
df_raw.columns

### Target Verteilung

In [ ]:
datasets = {
    "Raw": df_raw, 
    "Train": df_train, 
    "Val": df_val, 
    "Test": df_test
}
target_labels = {
    False : "Kein Versicherungsfall",
    True : "Versicherungsfall"
}

rows = []
for name, data in datasets.items():
    counts = data["target"].value_counts(normalize=True).sort_index()
    for value, share in counts.items():
        rows.append({"Datensatz": name, "Target": target_labels[value], "Anteil": share})

plot_df = pd.DataFrame(rows)
plot_df

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=plot_df, 
    x="Target", 
    y="Anteil", 
    hue="Datensatz", 
    ax=ax
)

ax.set_title("Verteilung der Zielvariable je Datensatz")
ax.set_xlabel("")
ax.set_ylabel("Anteil von Target")

for container in ax.containers:
    labels = [f"{v:.2%}" for v in container.datavalues]
    ax.bar_label(container, labels=labels, padding=2)

plt.tight_layout()

### Missing Values

In [ ]:
# sanity check wegen kaggle

(df_raw[feat_cols] == -1).sum().sum()

In [ ]:
duplikat_n = df_raw.duplicated().sum()
print(f"Anzahl doppelter Zeilen: {duplikat_n}")

In [ ]:
# MVs
miss_share = df_raw[feat_cols].isna().mean().sort_values(ascending=False)
miss_share = miss_share[miss_share > 0]
miss_count = df_raw[feat_cols].isna().sum()
miss_count = miss_count[miss_share.index]


fig, ax = plt.subplots(figsize=(10, max(3, len(miss_share))))
sns.barplot(
    x=miss_share.values, 
    y=miss_share.index, 
    ax=ax, 
    orient="h",
    palette="YlOrRd_r",
    hue=miss_share.index
)
ax.set_xlabel("Anteil fehlender Werte")
ax.set_ylabel("Features mit fehlenden Werten")
ax.set_title("Fehlende Werte pro Spalte (nur Spalten mit Missing Values)")

plt.tight_layout()

for container in ax.containers:
    labels = [f"<0.01%" if v < 0.00001 else f"{v:.2%}" for v in container.datavalues]
    ax.bar_label(container, labels=labels, padding=2)

x_left, x_right = ax.get_xlim()
ax.set_xlim(x_left, x_right*1.025)

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(3, len(miss_share))))

ax.hlines(
    xmin=miss_share.values.min(),
    xmax=miss_share.values,
    y=miss_share.index,
    linewidth=4,
)
ax.plot(
    miss_share.values, 
    miss_share.index,
    "o"
)
ax.set_xscale("log")
ax.set_xlabel("Anteil fehlender Werte (log Skala)")
ax.set_ylabel("Features mit fehlenden Werten")
ax.set_title("Anteil fehlender Werte (log Skala der Anteile, Absolute Werte als Beschreibung)")

for share, count, label in zip(miss_share.values, miss_count.values, miss_share.index):
    ax.text(share * 1.15, label, f"{count:,}", va="center", fontsize=9)

x_left, x_right = ax.get_xlim()
ax.set_xlim(x_left, x_right*2)

In [ ]:
top_miss_cols = miss_share.head(4).index

mv_target_rate_dict = {}
for cols in top_miss_cols:
    is_miss = df_raw[cols].isna()
    rate = df_raw.groupby(is_miss)["target"].mean()
    rate.index = ["vorhanden", "fehlende Werte"]
    mv_target_rate_dict[cols] = rate

mv_target_rate_df = pd.DataFrame(mv_target_rate_dict).T
mv_target_rate_df["diff"] = (mv_target_rate_df["fehlende Werte"] - mv_target_rate_df["vorhanden"]).abs()
mv_target_rate_df 

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

mv_target_rate_df[["vorhanden", "fehlende Werte"]].plot(
    kind="bar",
    ax=ax
)

target_mean_rate = df_raw["target"].mean()

ax.axhline(
    target_mean_rate,
    linestyle="--",
    label=f"Gesamtrate {target_mean_rate:.2%}",
    color="orange"
)

ax.set_title("Target Rate nach Missing Status (Top 4)")
ax.set_xlabel("Feature")
ax.set_ylabel("Target Rate")
ax.set_ylim(0, 0.049)
ax.legend(bbox_to_anchor=(1, 1))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

for container in ax.containers[:2]:
    labels = [f"{v:.2%}" for v in container.datavalues]
    ax.bar_label(container, labels=labels, padding=2)

In [ ]:
miss_by_split = pd.DataFrame({
    "Train" : df_train[feat_cols].isna().mean(),
    "Val" : df_val[feat_cols].isna().mean(),
    "Test" : df_test[feat_cols].isna().mean()
})

miss_by_split = miss_by_split[miss_by_split.sum(axis=1) > 0]
miss_by_split["max abweichung"] = miss_by_split.max(axis=1) - miss_by_split.min(axis=1)
miss_by_split.sort_values("max abweichung", ascending=False)

### Cat Features

In [ ]:
kardinal = df_raw[cat_cols].nunique().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    x=kardinal.values,
    y=kardinal.index,
    ax=ax,
    orient="h"
)
ax.set_title("Kardinalität der _cat features")
ax.set_xlabel("Anzahl unterschiedlicher Kategorien")
ax.set_ylabel("")
plt.tight_layout()

for container in ax.containers:
    labels = [f"{v.astype(int)}" for v in container.datavalues]
    ax.bar_label(container, labels=labels, padding=2)

In [ ]:
df_raw["ps_car_11_cat"].value_counts(ascending=False)

In [ ]:
top_n = 20
counts_n = df_raw["ps_car_11_cat"].value_counts()
top = counts_n.head(top_n)
top["sonstige"] = counts_n.iloc[top_n:].sum()

fig, ax = plt.subplots(figsize=(10, 5))

top.sort_values().plot(
    kind="barh",
    ax=ax
)
ax.set_title(f"ps_car_11_cat: Top {top_n} Kategorien")
ax.set_xlabel("Anzahl Beobachtungen")
ax.set_ylabel("")

In [ ]:
stats_kard = (
    df_raw.groupby("ps_car_11_cat", observed=True)["target"]
    .agg(["mean", "count"])
)

fig, ax = plt.subplots(figsize=(10,5))

ax.scatter(
    stats_kard["count"], 
    stats_kard["mean"],
      alpha=0.6
)
ax.set_xscale("log")
ax.axhline(
    df_raw["target"].mean(),
    linestyle="--",
    label="Gesamt Druchschnitt"
)
ax.set_title("ps_car_11_cat : Kategorie Größe vs Target Rate")
ax.set_xlabel("Anzahl Beobachtungen in der Kategorie (log Skala)")
ax.set_ylabel("Target Rate der Kategorie")
ax.legend()

In [ ]:
high_kard_thresh = 18
low_kard_cols = kardinal[kardinal <= high_kard_thresh].index
high_kard_cols = kardinal[kardinal > high_kard_thresh].index

kard_cols_grid = 3
kard_rows_grid = int(np.ceil(len(low_kard_cols) / kard_cols_grid))

fig, axes = plt.subplots(kard_rows_grid, kard_cols_grid, figsize=(20,10))

achsen = axes.flatten()
overall_rate = df_raw["target"].mean()

for idx, cols in enumerate(low_kard_cols):
    rates = (
        df_raw
        .groupby(cols, observed=True)["target"]
        .mean()
        .sort_values(ascending=False)
    )
    rates.plot(
        kind="bar",
        ax=achsen[idx]
    )
    achsen[idx].axhline(
        overall_rate,
        linestyle="--",
        linewidth=2,
        color="red"
    )
    achsen[idx].set_title(cols)
    achsen[idx].set_xlabel("")
    achsen[idx].set_ylabel("")
    achsen[idx].tick_params(axis="x")

for n in range(len(low_kard_cols), len(achsen)):
    achsen[n].axis("off")

plt.tight_layout()


### Bin Features

In [ ]:
bin_one_share = df_raw[bin_cols].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(3, 0.5* len(bin_one_share))))

sns.barplot(
    x=bin_one_share.values,
    y=bin_one_share.index,
    ax=ax,
    orient="h",
)
ax.axvline(
    0.5, 
    color="red", 
    linestyle="--"
)
ax.set_title("Anteil 1 je _bin Feature (Vergleich 50%)")
ax.set_xlabel("Anteil Feature = 1")
ax.set_ylabel("")
ax.set_xlim(0, 0.71)

for container in ax.containers:
    labels = ["<0.01%" if v < 0.0001 else f"{v:.2%}" for v in container.datavalues]
    ax.bar_label(container, labels=labels)

plt.tight_layout()

In [ ]:
bin_target_rate = pd.DataFrame({
    cols : df_raw.groupby(cols, observed=True)["target"].mean() for cols in bin_cols
}).T
bin_target_rate.columns = ["target_rate_bei_0", "target_rate_bei_1"]
bin_target_rate["diff"] = (
    bin_target_rate["target_rate_bei_1"] - bin_target_rate["target_rate_bei_0"]
).abs()

bin_target_rate.sort_values("diff", ascending=False)

### Num Features

In [ ]:
df_raw[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T.round(3)

In [ ]:
def count_outlier(series):
    "außreißer count zurünk geben (außerhalb q1 und q3)"
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_thresh = q1 - 1.5 * iqr
    upper_thresh = q3 + 1.5 * iqr
    return ((series < lower_thresh) | (series > upper_thresh)).sum()

outlier_counts = df_raw[num_cols].apply(count_outlier)

outlier_summary = pd.DataFrame({
    "anzahl_ausreißer": outlier_counts,
    "anteil": outlier_counts / len(df_raw)
})

outlier_summary.sort_values("anzahl_ausreißer", ascending=False)

In [ ]:
fig, axes = plt.subplots(num_rows_grid, num_cols_grid, figsize=(3*num_cols_grid, 2*num_rows_grid))

achsen = axes.flatten()

for idx, cols in enumerate(num_cols):
    df_raw.boxplot(
        column=cols,
        ax=achsen[idx],
        flierprops=dict(
            marker="o",
            markersize=2,
            alpha=0.3,
        )
    )
    achsen[idx].set_title(cols)
    achsen[idx].set_xlabel("")

for n in range(len(num_cols), len(achsen)):
    achsen[n].axis("off")

plt.tight_layout()

In [ ]:
corr_num_target = (
    df_raw[num_cols]
    .assign(target=df_raw["target"].astype(float))
    .corr()["target"]
    .drop("target")
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(10, max(3, 0.5*len(corr_num_target))))

sns.barplot(
    x=corr_num_target.values,
    y=corr_num_target.index,
    ax=ax,
    orient="h"
)
ax.axvline(0, color="red")
ax.set_title("Korrelation numerischer Features mit Target")
ax.set_xlabel("Korrelation mit Target")
ax.set_ylabel("")
ax.set_xlim(min(corr_num_target)*1.3, max(corr_num_target)*1.2)

for container in ax.containers:
    labels = [f"{v:.3f}" for v in container.datavalues]
    ax.bar_label(container, labels=labels)

plt.tight_layout()

corr_num_target

### Inter Korrelationen

In [ ]:
fig, ax = plt.subplots(figsize=(20,10))

sns.heatmap(
    df_raw[bin_cols].astype(float).corr(),
    ax=ax,
    annot=True,
    fmt=".2f"
)
ax.set_title("Korrelation zwischen bin Features")

In [ ]:
fig, ax = plt.subplots(figsize=(20,10))

sns.heatmap(
    df_raw[num_cols].corr(),
    ax=ax,
    xticklabels=True,
    yticklabels=True,
    annot=True,
    fmt=".3f"
)
ax.set_title("Korrelationsheatmap: numerische Features")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()

In [ ]:
corr_num_matrix = df_raw[num_cols].corr()

corr_num_pairs = corr_num_matrix.where(
    np.triu(np.ones(corr_num_matrix.shape, dtype=bool), k=1)
).stack()

corr_num_pairs = corr_num_pairs[corr_num_pairs.abs() > 0.5].sort_values(key=abs, ascending=False)

corr_num_pairs_df = corr_num_pairs.reset_index()
corr_num_pairs_df.columns = ["feature_1", "feature_2", "corr"]
corr_num_pairs_df

In [ ]:
#calc scheint kaum korrs mit target aufzuweisen deshalb nochmal
calc_num_cols = [cols for cols in num_cols if cols.startswith("ps_calc_")]

calc_num_corr_target = corr_num_target[calc_num_cols].sort_values(key=abs, ascending=False)

print("Korrelation calc_ num features Target:")
calc_num_corr_target


In [ ]:
fig, ax = plt.subplots(figsize=(20,10))

sns.heatmap(
    df_raw[calc_num_cols].corr(),
    ax=ax,
    xticklabels=True,
    yticklabels=True,
    annot=True,
    fmt=".3f"
)
ax.set_title("Korrelationsheatmap: numerische calc_ Features")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()

### Gruppen

In [ ]:
prefix_groups = {}
for prefix in {"ind", "reg", "car", "calc"}:
    prefix_groups[prefix] = [cols for cols in feat_cols if cols.startswith(f"ps_{prefix}_")]
    print(f"ps_{prefix}_: {len(prefix_groups[prefix])} Columns")

In [ ]:
for prefix, cols in prefix_groups.items():
    cols_num = [col for col in cols if col in num_cols]
    cols_bin = [col for col in cols if col in bin_cols]
    cols_cat = [col for col in cols if col in cat_cols]

    print(f"Für {prefix} :")
    print(f"num: {len(cols_num)}")
    print(f"bin: {len(cols_bin)}")
    print(f"cat: {len(cols_cat)}")

    compatible_cols = cols_num + cols_bin
    if len(compatible_cols) >= 2:
        korr = df_raw[compatible_cols].astype(float).corr()
        max_korr = korr.where(np.triu(np.ones(korr.shape, dtype=bool), k=1)).abs().max().max()
        print(f"max korr num/bin: {max_korr:.3f}" if pd.notna(max_korr) else "  nur 1 Feature")
    
    if len(cols_cat) >= 2:
        print(f"{len(cols_cat)} cat features: Cramers V prüfen")

In [ ]:
result_pairs = []

for prefix in ["reg", "ind", "car"]:
    cols = prefix_groups[prefix]
    compatible_cols = [col for col in cols if col in num_cols + bin_cols]
    korr = df_raw[compatible_cols].astype(float).corr()

    pairs = korr.where(np.triu(np.ones(korr.shape, dtype=bool), k=1)).stack()
    pairs = pairs[pairs.abs() > 0.5]

    for (col1, col2), v in pairs.items():
        result_pairs.append({
            "prefix": f"ps_{prefix}_", 
            "feature_1": col1,
            "feature_2": col2,
            "korr": v
        })

result_pairs_df = pd.DataFrame(result_pairs).sort_values("korr", key=abs, ascending=False)
result_pairs_df

In [ ]:
def cramers_v(x,y):
    """eigene cramer v über kreuztabelle und chi2"""
    conti = pd.crosstab(x,y)
    chi_2 = chi2_contingency(conti)[0]
    n = conti.sum().sum()
    r, k = conti.shape
    return np.sqrt(chi_2 / (n * (min(r,k) - 1)))

reslut_cramer = []

for prefix in ["ind", "car"]:
    cols_cat = [col for col in prefix_groups[prefix] if col in cat_cols]
    for col1, col2 in itertools.combinations(cols_cat, 2):
        v = cramers_v(df_raw[col1], df_raw[col2])
        reslut_cramer.append({
            "prefix": f"ps_{prefix}_", 
            "feature_1": col1,
            "feature_2": col2,
            "cramers_v": v
        })

cramers_df = pd.DataFrame(reslut_cramer).sort_values("cramers_v", ascending=False)
cramers_df

## Preprocessing Experimente

### Preproc Linear
Die linearen Modelle verwenden immer eine Anpassung des selben Preprocessings, ich definiere den column transformer hier vor

In [ ]:
def preproc_linear(num_cols_use=None, bin_cols_use=None, drop_first=False):
    """angabe welche cols des preprocessings genutzt werden sollen"""

    return make_column_transformer(
        (
            make_pipeline(
                SimpleImputer(strategy="median", add_indicator=True),
                StandardScaler()
            ), num_cols if num_cols_use is None else num_cols_use
        ),
        (
            OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first" if drop_first else None),
            low_kard_cols
        ),
        (
            TargetEncoder(cv=5, random_state=RANDOM_STATE),
            high_kard_cols
        ),
        (
            "passthrough",
            bin_cols if bin_cols_use is None else bin_cols_use
        )
    )

### Logistische Regression

|Index|Spezifikation|
|---|---|
|L_lr_01|Baseline: Standardpreprocessing, Standardparameter|
|L_lr_02|class_weight="balanced" gegen die Klassenimbalance|
|L_lr_03|zusätzlich drop="first" im One-Hot-Encoding|
|L_lr_04|zusätzlich Drop von `ps_ind_06_bin|
|L_lr_05|Drop stark korrelierter Features (ps_reg_03, ps_ind_14, ps_ind_18_bin)|
|L_lr_06|Drop der ps_calc_* Gruppe|

In [ ]:
res_lr = []
tt_lr = {}

In [ ]:
num_cols_v5 = [c for c in num_cols if c not in ("ps_reg_03", "ps_ind_14")]
bin_cols_v5 = [c for c in bin_cols if c != "ps_ind_18_bin"]
bin_cols_one_drop = [c for c in bin_cols if c != "ps_ind_06_bin"]

In [ ]:
logreg_exp = {
    "L_lr_01": make_pipeline(
        preproc_linear(),
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ),
    "L_lr_02": make_pipeline(
            preproc_linear(),
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
    ),
    "L_lr_03": make_pipeline(
            preproc_linear(drop_first=True),
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
    ),
    "L_lr_04": make_pipeline(
            preproc_linear(drop_first=True, bin_cols_use=bin_cols_one_drop),
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
    ),
    "L_lr_05": make_pipeline(
            preproc_linear(drop_first=True, bin_cols_use=bin_cols_v5, num_cols_use=num_cols_v5),
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
    ),
    "L_lr_06": make_pipeline(
            preproc_linear(drop_first=True, bin_cols_use=bin_cols_no_calc, num_cols_use=num_cols_no_calc),
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
    )
}

In [ ]:
for name, modell in logreg_exp.items():
    start = time.time()
    modell.fit(x_train, y_train)
    tt_lr[name] = time.time() - start

    res_lr.append(
        eval_modell(name, modell, x_train, y_train, x_val, y_val, tt_lr[name])
    )

pd.DataFrame(res_lr)

### XGBoost

|Index | Spezifikation |
|---|---|
|L_xgb_00_nativ|Kategorische Features nativ, keinerlei Kodierung|
|L_xgb_01|One-Hot und Target Encoding vorgeschaltet, Standardparameter|
|L_xgb_02|Native Verarbeitung, mehr Kapazität (n_estimators=1000, lr=0.05, Early Stopping)|
|L_xgb_02MK2|n_estimators=5000, lr=0.03, ohne enable_categorical|
|L_xgb_03|wie MK2, mit enable_categorical|
|L_xgb_04|Klassenimbalance über `scale_pos_weight|
|L_xgb_05|Drop der ps_calc_* Gruppe|
|L_xgb_06|Target Encoding für `ps_car_11_cat|
|L_xgb_07|Target Encoding und calc-Drop|

In [ ]:
res_xgb = []
tt_xgb = {}

In [ ]:
imbalance = (y_train == 0).sum() / (y_train == 1).sum()

targ_enc_06 = TargetEncoder(cv=5, random_state=RANDOM_STATE)

x_train_06 = x_train.copy()
x_val_06 = x_val.copy()

x_train_06["ps_car_11_cat"] = targ_enc_06.fit_transform(
    x_train[high_kard_cols], y_train
).astype("float32")
x_val_06["ps_car_11_cat"] = targ_enc_06.transform(
    x_val[high_kard_cols]
).astype("float32")

targ_enc_06_no_calc = TargetEncoder(cv=5, random_state=RANDOM_STATE)

x_train_06_no_calc = x_train[feat_cols_no_calc].copy()
x_val_06_no_calc = x_val[feat_cols_no_calc].copy()

x_train_06_no_calc["ps_car_11_cat"] = targ_enc_06_no_calc.fit_transform(
    x_train[high_kard_cols], y_train
).astype("float32")
x_val_06_no_calc["ps_car_11_cat"] = targ_enc_06_no_calc.transform(
    x_val[high_kard_cols]
).astype("float32")

In [ ]:
xgb_exp = {
    "L_xgb_00": XGBClassifier(
        booster="gbtree",
        random_state=RANDOM_STATE,
        eval_metric="auc",
        tree_method="hist"
    ),
    "L_xgb_01": make_pipeline(
        make_column_transformer(
            ("passthrough", num_cols),
            (OneHotEncoder(handle_unknown="ignore", sparse_output=False), low_kard_cols),
            (TargetEncoder(cv=5, shuffle=True, random_state=RANDOM_STATE), high_kard_cols),
            ("passthrough", bin_cols)
        ),
        XGBClassifier(
            booster="gbtree",
            random_state=RANDOM_STATE,
            eval_metric="auc",
            tree_method="hist"
        )
    ),
    "L_xgb_02": XGBClassifier(
        booster="gbtree",
        random_state=RANDOM_STATE,
        eval_metric="auc",
        tree_method="hist",
        n_estimators=1000,
        learning_rate=0.05,
        early_stopping_rounds=50
    ),
    "L_xgb_02MK2": XGBClassifier(
        booster="gbtree",
        random_state=RANDOM_STATE,
        eval_metric="auc",
        tree_method="hist",
        n_estimators=5000,
        learning_rate=0.03,
        early_stopping_rounds=100
    ),
    "L_xgb_03": XGBClassifier(
        booster="gbtree",
        random_state=RANDOM_STATE,
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=True,
        n_estimators=5000,
        learning_rate=0.03,
        early_stopping_rounds=100
    ),
    "L_xgb_04": XGBClassifier(
        booster="gbtree",
        random_state=RANDOM_STATE,
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=True,
        n_estimators=5000,
        learning_rate=0.03,
        early_stopping_rounds=100,
        scale_pos_weight=imbalance
    ),
    "L_xgb_05" = XGBClassifier(
        booster="gbtree",
        random_state=RANDOM_STATE,
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=True,
        n_estimators=5000,
        learning_rate=0.03,
        early_stopping_rounds=100
    ),
    "L_xgb_06" = XGBClassifier(
        booster="gbtree",
        random_state=RANDOM_STATE,
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=True,
        n_estimators=5000,
        learning_rate=0.03,
        early_stopping_rounds=100
    ),
    "L_xgb_07" = XGBClassifier(
        booster="gbtree",
        random_state=RANDOM_STATE,
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=True,
        n_estimators=5000,
        learning_rate=0.03,
        early_stopping_rounds=100
    )
}

In [ ]:
for name, modell in xgb_exp.items():
    if name not in ["L_xgb_05, ", "L_xgb_06", "L_xgb_07"]:
        start = time.time()
        modell.fit(
            x_train, y_train,
            eval_set=[(x_train, y_train),(x_val, y_val)]
        )
        tt_xgb[name] = time.time() - start
        res_xgb.append(
            eval_modell(name, modell, x_train, y_train, x_val, y_val, tt_xgb[name])
        )
    elif name == "L_xgb_05":
        start = time.time()
        modell.fit(
            x_train_no_calc, y_train,
            eval_set=[(x_train_no_calc, y_train),(x_val_no_calc, y_val)]
        )
        tt_xgb[name] = time.time() - start
        res_xgb.append(
            eval_modell(name, modell, x_train_no_calc, y_train, x_val_no_calc, y_val, tt_xgb[name])
        )
    elif name == "L_xgb_06":
        start = time.time()
        modell.fit(
            x_train_06, y_train,
            eval_set=[(x_train_06, y_train),(x_val_06, y_val)]
        )
        tt_xgb[name] = time.time() - start
        res_xgb.append(
            eval_modell(name, modell, x_train_06, y_train, x_val_06, y_val, tt_xgb[name])
        )
    elif name == "L_xgb_07":
        start = time.time()
        modell.fit(
            x_train_06_no_calc, y_train,
            eval_set=[(x_train_06_no_calc, y_train),(x_val_06_no_calc, y_val)]
        )
        tt_xgb[name] = time.time() - start
        res_xgb.append(
            eval_modell(name, modell, x_train_06_no_calc, y_train, x_val_06_no_calc, y_val, tt_xgb[name])
        )

pd.DataFrame(res_xgb)

### Support Vector Machine

|Index|Spezifikation|
|---|---|
|L_svm_01|Baseline, Preprocessing wie bei der logistischen Regression|
|L_svm_02|dual=False (primales Problem, da n_samples > n_features)|
|L_svm_03|zusätzlich class_weight="balanced"|
|L_svm_04|drop="first" im One-Hot|
|L_svm_04MK2|zusätzlich Drop von ps_ind_06_bin|
|L_svm_05|zusätzlicher StandardScaler über die gesamte Designmatrix|
|L_svm_06|Drop der ps_calc_* Gruppe|

In [ ]:
def eval_modell_svm(mod_idx, model, x_train, y_train, x_val, y_val, train_time):
    """AUC auf Train und Val für svm"""
    auc_train = roc_auc_score(y_train, model.decision_function(x_train))
    auc_val = roc_auc_score(y_val, model.decision_function(x_val))
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_val": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "trainingszeit": train_time
    }

In [ ]:
res_svm = []
tt_svm = {}

In [ ]:
svm_varianten = {
    "L_svm_01": make_pipeline(
        preproc_linear(),
        LinearSVC(max_iter=1000, random_state=RANDOM_STATE)
    ),
    "L_svm_02": make_pipeline(
        preproc_linear(),
        LinearSVC(max_iter=1000, random_state=RANDOM_STATE, C=1.0, dual=False)
    ),
    "L_svm_03": make_pipeline(
        preproc_linear(),
        LinearSVC(max_iter=1000, random_state=RANDOM_STATE, C=1.0, dual=False, class_weight="balanced")
    ),
    "L_svm_04": make_pipeline(
        preproc_linear(drop_first=True),
        LinearSVC(max_iter=1000, random_state=RANDOM_STATE, C=1.0, dual=False)
    ),
    "L_svm_04MK2": make_pipeline(
        preproc_linear(drop_first=True, bin_cols_use=bin_cols_one_drop),
        LinearSVC(max_iter=1000, random_state=RANDOM_STATE, C=1.0, dual=False)
    ),
    "L_svm_05": make_pipeline(
        preproc_linear(),
        StandardScaler(),
        LinearSVC(max_iter=1000, random_state=RANDOM_STATE, C=1.0, dual=False)
    ),
    "L_svm_06": make_pipeline(
        preproc_linear(num_cols_use=num_cols_no_calc, bin_cols_use=bin_cols_no_calc),
        LinearSVC(max_iter=1000, random_state=RANDOM_STATE, C=1.0, dual=False)
    ),
}

In [ ]:
for name, model in svm_varianten.items():
    start = time.time()
    model.fit(x_train, y_train)
    tt_svm[name] = time.time() - start
    res_svm.append(eval_modell(name, model, x_train, y_train, x_val, y_val, tt_svm[name]))

pd.DataFrame(res_svm)

### Catboost
L_cb_01 & L_cb_02 waren versuche, bei denen ich auf fehler stieß (Missing Values in cat, Fehlerhafte Pipeline)

|Index|Spezifikation|
|---|---|
|L_cb_03|Reine Defaultparameter, manuelle Vorverarbeitung|
|L_cb_05|Feste Rahmenparameter: Early Stopping, use_best_model, iterations=5000|
|L_cb_07|wie L_cb_05, aber boosting_type="Ordered"|
|L_cb_08|wie L_cb_05, ohne die ps_calc_* Gruppe|

## Hyperparameter Optimierung

### CatBoost

### XGBoost

### HistGradientBoosting

## PCA

## Validierungsset Evaluation

## Testset Evaluation

## Notizen